# 04 — Regresión: goles del equipo local (CRISP-DM: fases 4–5)

Target continuo: `home_team_goal`. Mismas cuotas como features.

**Siguiente:** `05_explicabilidad_evaluacion.ipynb`.

In [ ]:
%load_ext kedro.ipython

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "raw" / "database.sqlite"
assert (PROJECT_ROOT / "pyproject.toml").is_file(), (
    "Abre este notebook desde la raíz del proyecto con `kedro jupyter lab`."
)
assert DB_PATH.is_file(), (
    "No existe data/raw/database.sqlite. Ejecuta: `python scripts/bootstrap_data.py`."
)
print(f"Proyecto: {PROJECT_ROOT}")
print(f"SQLite: {DB_PATH}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler

match = catalog.load("match")
goal_diff = match["home_team_goal"] - match["away_team_goal"]
outcome = np.select(
    [goal_diff > 0, goal_diff == 0, goal_diff < 0],
    [2, 1, 0],
)
feat_cols = ["B365H", "B365D", "B365A"]
df = match[feat_cols].assign(outcome=outcome, home_team_goal=match["home_team_goal"]).dropna()
X = df[feat_cols]
y = df["home_team_goal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
reg_models = {
    "Ridge": SkPipeline(
        [("scale", StandardScaler()), ("reg", Ridge(alpha=1.0))]
    ),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=200,
        max_depth=16,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=1,
    ),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(
        max_iter=200,
        max_depth=6,
        learning_rate=0.08,
        random_state=42,
    ),
}

rows, fitted_reg = [], {}
for name, est in reg_models.items():
    est.fit(X_train, y_train)
    pred = est.predict(X_test)
    fitted_reg[name] = est
    rows.append(
        {
            "modelo": name,
            "mae": mean_absolute_error(y_test, pred),
            "rmse": mean_squared_error(y_test, pred) ** 0.5,
            "r2": r2_score(y_test, pred),
        }
    )

resultados_reg = pd.DataFrame(rows).sort_values("r2", ascending=False).reset_index(drop=True)
resultados_reg

In [ ]:
# Residuos del mejor modelo (y_true - y_pred)
best = resultados_reg.iloc[0]["modelo"]
pred = fitted_reg[best].predict(X_test)
resid = y_test.to_numpy() - pred

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(pred, y_test, alpha=0.25, s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=1)
axes[0].set_xlabel("Predicho")
axes[0].set_ylabel("Real")
axes[0].set_title(f"Real vs predicho — {best}")

axes[1].hist(resid, bins=40, edgecolor="white")
axes[1].set_title("Residuos (real − predicho)")
plt.tight_layout()
plt.show()

print("MAE en goles ≈ error típico en la escala del marcador.")